In [2]:
# importaçao de bibliotecas
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import time
import pandas as pd
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

In [ ]:
tabela_book = pd.read_excel("Entendendo o desafio - Produtos.xlsx")
tabela_book["link"] = str(tabela_book['link'])
tabela_book["preco"] = str(tabela_book['preco'])
display(tabela_book.info())

<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   nome       6 non-null      str  
 1   autor      6 non-null      str  
 2   categoria  6 non-null      str  
 3   preco      6 non-null      str  
 4   link       6 non-null      str  
dtypes: str(5)
memory usage: 1.5 KB


None

In [4]:

opcoes = Options()

opcoes.add_argument(r"--user-data-dir=C:\SeleniumProfile") 

driver = webdriver.Chrome(options=opcoes)

wait = WebDriverWait(driver, 10)


# Função pesquisar no Gutenberg

In [50]:
def pesquisa_gutenberg(livro, autor):
    site = r"https://www.gutenberg.org"

    driver.get(site)

    lista_nome_autor = autor.split(" ")
    try:
        barra = wait.until(EC.presence_of_element_located((By.CLASS_NAME, "search-input"))).send_keys(livro, Keys.ENTER)

        lista_livros = wait.until(EC.presence_of_all_elements_located((By.CLASS_NAME, "booklink")))

        link = None

        for resultado in lista_livros:
            texto = resultado.text
            if livro.lower() in texto.lower():
                if all(palavra.lower() in texto.lower() for palavra in lista_nome_autor):
                    link = resultado.find_element(By.CLASS_NAME, 'link').get_attribute("href")
                    break

        if link:
            preco = "0"
            return link, preco
        else:
            preco = None
            return link, preco
    except TimeoutException:
        return None, None


# Função pesquisar no Booktoscrape


In [52]:

def pesquisa_toscrape(livro, categoria):
    driver.get(r"https://books.toscrape.com/index.html")

    sumario = wait.until(EC.presence_of_element_located((By.CLASS_NAME, "nav-list")))

    book = livro

    categoria = categoria

    sumario_nome = sumario.find_elements(By.TAG_NAME, 'a')
    for item in sumario_nome:
        title = item.text
        if categoria.lower() in title.lower():
            item.click()
            break

    lista_livros = wait.until(EC.presence_of_all_elements_located((By.CLASS_NAME, "product_pod")))

    link = None
    preco = None

    for livro in lista_livros:
        texto = livro.text
        if book.lower() in texto.lower():
            h3 = livro.find_element(By.TAG_NAME, "h3")
            link = h3.find_element(By.TAG_NAME, "a").get_attribute("href")
            break
    if link:
        preco = livro.find_element(By.CLASS_NAME, "price_color").text
        return link, preco
    else:
        return None, None


In [55]:
for linha in tabela_book.index:
    livro = tabela_book.loc[linha, "nome"]
    autor = tabela_book.loc[linha, 'autor']
    categoria = tabela_book.loc[linha, 'categoria']

    link1, preco1 = pesquisa_gutenberg(livro, autor)
    link2, preco2 = pesquisa_toscrape(livro, categoria)
    if link1:
        tabela_book.loc[linha, "link"] = (link1)
        tabela_book.loc[linha, 'preco'] = (preco1)
    elif link2:
        tabela_book.loc[linha, "link"] = (link2)
        tabela_book.loc[linha, 'preco'] = (preco2)
    else:
        tabela_book.loc[linha, "link"] = 'not found'
        tabela_book.loc[linha, 'preco'] = "not found"

tabela_book.to_excel("Tabela_Atualizada.xlsx")